In [ ]:
!pip install albumentations pillow faiss-cpu tqdm transformers torch

In [ ]:
!pip install scikit-learn


In [ ]:
import time
import cv2
import os
import numpy as np
import faiss
import torch
import pickle
from tqdm import tqdm
from PIL import Image
from transformers import ViTModel, AutoImageProcessor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
# 🆕 Import Albumentations để Augmentation
import albumentations as A

class DinoFaceRecognition:
    def __init__(self, src_dir=None):
        self.src_dir = src_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model_name = "facebook/dino-vits8"
        self.model = ViTModel.from_pretrained(self.model_name).to(self.device)
        self.model.eval()
        self.processor = AutoImageProcessor.from_pretrained(self.model_name)

        self.index = None
        self.labels = []
        self.class_to_id = {}
        self.id_to_class = {}

        # 🆕 Define augmentation pipeline
        self.augmentor = A.Compose([
            A.OneOf([
                A.CoarseDropout(max_holes=1, max_height=32, max_width=32, p=1.0),
                A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            ], p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.HorizontalFlip(p=0.5)
        ])

    def augment_image(self, image, num_augments=4):
        """Augment an image num_augments times."""
        augmented_images = []
        for _ in range(num_augments):
            augmented = self.augmentor(image=image)
            augmented_images.append(augmented['image'])
        return augmented_images

    def load_data(self, faiss_index_path=None, metadata_path=None):
        if faiss_index_path and metadata_path:
            self.index = faiss.read_index(faiss_index_path)
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
                self.labels = metadata['labels']
                self.class_to_id = metadata['label_to_id']
                self.id_to_class = {v: k for k, v in self.class_to_id.items()}

    def extract_features(self, images, normalize=True):
        try:
            if not isinstance(images, list):
                images = [images]
            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            if normalize:
                embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
            return embeddings
        except Exception as e:
            print(f"Error in feature extraction: {e}")
            return None


    def build_and_save_faiss_index(self, features, save_path=None):
        # Nếu index chưa được tạo, tạo mới
        if self.index is None:
            try:
                if features is None or features.size == 0:
                    raise ValueError("No features provided for FAISS index")

                features = features.astype('float16')  # 🆕 Quantize to float16
                dim = features.shape[1]
                self.index = faiss.IndexFlatIP(dim)  # FAISS index using Inner Product (Cosine Similarity)
                self.index.add(features)  # Thêm features vào index

                # Lưu index vào file
                if save_path:
                    faiss.write_index(self.index, save_path + '.faiss')
                    with open(f"{save_path}_metadata.pkl", "wb") as f:
                        pickle.dump({
                            "labels": self.labels,
                            "label_to_id": self.class_to_id
                        }, f)
                    print(f"FAISS index saved to {save_path}.faiss")

            except Exception as e:
                print(f"Error building/saving FAISS index: {e}")
                raise
        else:
            # Nếu index đã tồn tại, chỉ cần thêm các features mới vào
            try:
                if features is None or features.size == 0:
                    raise ValueError("No features provided for FAISS index")

                features = features.astype('float16')  # 🆕 Quantize to float16
                self.index.add(features)  # Thêm features vào index hiện có

                # Lưu lại index sau khi thêm các features
                if save_path:
                    faiss.write_index(self.index, save_path + '.faiss')
                    with open(f"{save_path}_metadata.pkl", "wb") as f:
                        pickle.dump({
                            "labels": self.labels,
                            "label_to_id": self.class_to_id
                        }, f)
                    print(f"FAISS index updated and saved to {save_path}.faiss")

            except Exception as e:
                print(f"Error adding features to FAISS index: {e}")
                raise

    def train_in_batches(self, faiss_index_path=None, num_augments=4, batch_size=10):
        """Train in batches to prevent RAM overflow (process 10 labels at a time)."""
        if not self.src_dir:
            raise ValueError("Source directory (src_dir) not specified")

        images = []
        labels = []
        all_class_names = os.listdir(self.src_dir)

        total_labels = len(all_class_names)
        for i in tqdm(range(0, total_labels, batch_size)):
            # Lấy 10 labels mỗi lần
            batch_class_names = all_class_names[i:i + batch_size]
            images = []
            labels = []

            for class_name in batch_class_names:
                class_dir = os.path.join(self.src_dir, class_name)
                if os.path.isdir(class_dir):
                    for img_name in os.listdir(class_dir):
                        img_path = os.path.join(class_dir, img_name)
                        if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                            img = cv2.imread(img_path)
                            if img is None:
                                continue
                            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            images.append(img)
                            labels.append(class_name)
                            # Add augmented images
                            augmented_imgs = self.augment_image(img, num_augments=num_augments)
                            images.extend(augmented_imgs)
                            labels.extend([class_name] * num_augments)

            if not images:
                raise ValueError(f"No valid images found for labels {batch_class_names}")

            # Trích xuất đặc trưng cho batch hiện tại
            unique_labels = sorted(set(labels))
            self.class_to_id = {label: i for i, label in enumerate(unique_labels)}
            self.id_to_class = {i: label for label, i in self.class_to_id.items()}
            self.labels = [self.class_to_id[label] for label in labels]

            features = self.extract_features(images)
            if features is None:
                raise ValueError("Feature extraction failed for batch")

            # Lưu vào FAISS
            self.build_and_save_faiss_index(features, save_path=faiss_index_path)
            print(f"label {i} processed / total {total_labels} labels")
            # Sau khi lưu FAISS, có thể làm việc tiếp với batch tiếp theo


    def recognize_face(self, query_img, threshold=0.9, top_k=1):
        if query_img is None or query_img.size == 0:
            return [("Invalid Image", 0.0)]
        try:
            query_img = cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB)
            query_embed = self.extract_features([query_img])
            if query_embed is None:
                return [("No Features Extracted", 0.0)]

            similarities, indices = self.index.search(query_embed.astype('float16'), k=top_k)

            results = []
            for i in range(top_k):
                pred_id = self.labels[indices[0][i]]
                similarity = float(similarities[0][i])
                user_name = self.id_to_class.get(pred_id, "Unknown")

                if similarity > threshold:
                    results.append((user_name, similarity))
                else:
                    results.append(("Unknown", similarity))
            return results
        except Exception as e:
            print(f"Error in face recognition: {e}")
            return [("Error", 0.0)]

    def print_faiss_size(faiss_file_path):
        if os.path.exists(faiss_file_path):
            size_in_bytes = os.path.getsize(faiss_file_path)
            size_in_mb = size_in_bytes / (1024 * 1024)
            print(f"[INFO] FAISS Index size: {size_in_mb:.2f} MB ({size_in_bytes} bytes)")
        else:
            print(f"[WARN] FAISS file not found: {faiss_file_path}")
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

    def evaluate_on_testset(self, test_dir, threshold=0.9, top_k=1):
        """
        Đánh giá mô hình nhận diện khuôn mặt trên tập test.

        Args:
            test_dir (str): Thư mục chứa các thư mục con (label) với ảnh test.
            threshold (float): Ngưỡng xác định nhận diện đúng.
            top_k (int): Số lượng kết quả top-k để đánh giá.

        Returns:
            dict: Kết quả gồm Accuracy, F1 Score, ROC AUC.
        """
        true_labels = []
        predicted_labels = []
        scores = []

        class_names = sorted(os.listdir(test_dir))
        for label in class_names:
            class_path = os.path.join(test_dir, label)
            if not os.path.isdir(class_path):
                continue
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_path, img_name)
                    img = cv2.imread(img_path)
                    if img is None:
                        continue
                    result = self.recognize_face(img, threshold=threshold, top_k=top_k)[0]
                    pred_label, score = result
                    true_labels.append(label)
                    predicted_labels.append(pred_label)
                    scores.append(score)

        # Tính toán các chỉ số
        acc = accuracy_score(true_labels, predicted_labels)
        f1 = f1_score(true_labels, predicted_labels, average='macro')

        # Tính ROC AUC nếu có đầy đủ nhãn và điểm
        label_list = sorted(set(true_labels + predicted_labels))
        label_to_idx = {label: idx for idx, label in enumerate(label_list)}
        y_true = [label_to_idx[l] for l in true_labels]
        y_score_matrix = np.zeros((len(scores), len(label_list)))
        for i, (pl, sc) in enumerate(zip(predicted_labels, scores)):
            if pl in label_to_idx:
                y_score_matrix[i][label_to_idx[pl]] = sc

        try:
            roc_auc = roc_auc_score(y_true, y_score_matrix, multi_class='ovr')
        except:
            roc_auc = "Cannot compute ROC AUC (possibly not enough classes)"

        return {
            "Accuracy": acc,
            "F1 Score (macro)": f1,
            "ROC AUC": roc_auc
        }


In [ ]:

dino = DinoFaceRecognition(src_dir='train')
# 🆕 Train with augmentation
dino.train_in_batches(faiss_index_path='faiss_index', num_augments=4, batch_size=1)

In [ ]:
dino.print_faiss_size(faiss_file_path='faiss_index.faiss')
#|%%--%%| <AGMxKnSMeM|mJYsVXolyl>
if dino.index is not None:
    print(f"[INFO] FAISS Index contains {dino.index.ntotal} vectors.")
else:
    print("[WARN] FAISS index not initialized.")

In [ ]:

dino.load_data('faiss_index.faiss', 'faiss_index_metadata.pkl')
results = dino.evaluate_on_testset(test_dir='test')
print(results)
